数学基础：
向量内积（Inner Product）


# 跳元模型（Skip-gram）概率计算

给定中心词 $w_c$ 与上下文词 $w_o$，跳元模型的条件概率为：

$$P(w_o \mid w_c) = \frac{\exp(\mathbf{u}_o^\top \mathbf{v}_c)}{\sum_{i \in \mathcal{V}} \exp(\mathbf{u}_i^\top \mathbf{v}_c)}$$

其中：
- $\mathbf{v}_c$：中心词 $w_c$ 的向量表示（中心嵌入）
- $\mathbf{u}_o$：上下文词 $w_o$ 的向量表示（外部嵌入）
- $\mathcal{V}$：词表

下面用 **scipy.special.softmax** 对小样本集计算该概率。

In [65]:
import numpy as np
from scipy.special import softmax
from itertools import chain

## 1. 小样本语料与词表

使用简单句子构建（中心词 + 窗口内上下文），便于手算对照。

In [66]:
# 小样本：每行为一句，词用空格分隔（类似图中“中心词-上下文”结构）
corpus = [
    "我 喜欢 学习 机器 学习",
    "机器 学习 很 有趣",
    "我 喜欢 编程",
]
window_size = 2  # 中心词左右各取 window_size 个词作为上下文

def tokenize(sentences):
    return [s.split() for s in sentences]

def build_vocab(sentences):
    """sentences: 分词后的列表，即 list of list of words，如 [['我','喜欢','学习'], ...]"""
    words = sorted(set(chain.from_iterable(sentences)))
    w2i = {w: i for i, w in enumerate(words)}
    i2w = {i: w for w, i in w2i.items()}
    return w2i, i2w, words

def get_center_context_pairs(tokenized, window_size):
    """收集 (中心词, 上下文词) 对"""
    pairs = []
    for sent in tokenized:
        for i, center in enumerate(sent):
            for j in range(max(0, i - window_size), min(len(sent), i + window_size + 1)):
                if i != j:
                    pairs.append((center, sent[j]))
    return pairs

tokenized = tokenize(corpus)
w2i, i2w, vocab = build_vocab(tokenized)  # 必须用分词后的列表，否则会把字符串按字符拆开
pairs = get_center_context_pairs(tokenized, window_size)

print("tokenization V:", vocab)
print("tokenization size |V|:", len(vocab))
print("\n(中心词, 上下文词) 对样本:", pairs[:12])

tokenization V: ['喜欢', '学习', '很', '我', '有趣', '机器', '编程']
tokenization size |V|: 7

(中心词, 上下文词) 对样本: [('我', '喜欢'), ('我', '学习'), ('喜欢', '我'), ('喜欢', '学习'), ('喜欢', '机器'), ('学习', '我'), ('学习', '喜欢'), ('学习', '机器'), ('学习', '学习'), ('机器', '喜欢'), ('机器', '学习'), ('机器', '学习')]


## 2. 嵌入与 P(w_o | w_c)（scipy softmax）

### 与代码对应的数学表示

**符号约定**
- 词表 $\mathcal{V}$，大小 $|V|$；词 $w$ 的索引 $i = \texttt{w2i}[w]$。
- 嵌入维度 $d = \texttt{embed\_dim}$。


In [67]:
embed_dim = 4
np.random.seed(42)
V = len(vocab)

**矩阵与向量**

中心词嵌入矩阵（每行为词 $i$ 作为中心词时的向量 $\mathbf{v}_i$）：
$$\mathbf{V}_{\mathrm{center}} = \begin{bmatrix} \mathbf{v}_1 \\ \mathbf{v}_2 \\ \vdots \\ \mathbf{v}_{|V|} \end{bmatrix} \in \mathbb{R}^{|V| \times d}$$

上下文词嵌入矩阵（每行为词 $i$ 作为上下文时的向量 $\mathbf{u}_i$）：
$$\mathbf{U}_{\mathrm{outside}} = \begin{bmatrix} \mathbf{u}_1 \\ \mathbf{u}_2 \\ \vdots \\ \mathbf{u}_{|V|} \end{bmatrix} \in \mathbb{R}^{|V| \times d}$$

训练时，修改的就是这个Vcenter和Uoutside

In [68]:
# 中心词嵌入 v_c: 词 w 作为中心词时的向量
# 是一个向量组组成的矩阵，每个向量代表一个词作为中心词时的向量
V_center = np.random.randn(V, embed_dim) * 0.1
# 上下文词嵌入 u_i: 词 w 作为上下文时的向量
# 是一个向量组组成的矩阵，每个向量代表一个词作为上下文时的向量
U_outside = np.random.randn(V, embed_dim) * 0.1

**给定中心词 $w_c$ 时的计算**
1. 取中心词向量：$\mathbf{v}_c = \mathbf{V}_{\mathrm{center}}[\texttt{w2i}[w_c]] \in \mathbb{R}^d$。
2. Logits（未归一化分数）：
   $$\mathbf{z} = \mathbf{U}_{\mathrm{outside}} \, \mathbf{v}_c \in \mathbb{R}^{|V|}, \qquad z_i = \mathbf{u}_i^\top \mathbf{v}_c.$$
3. 概率分布 P(·|w_c)：
   $$P(w_o \mid w_c) = \frac{\exp(z_o)}{\sum_{i \in \mathcal{V}} \exp(z_i)} = \texttt{softmax}(\mathbf{z})\,[\texttt{w2i}[w_o]].$$
   整条向量 $\texttt{softmax}(\mathbf{z})$ 即为代码返回的 P(·|w_c) 数组。

In [ ]:
# 核心方法，softmax回归
def prob_context_given_center(v_c, U_outside_mat):
    """给定中心词向量 v_c，计算 P(·|w_c)。返回形状 (|V|,) 的概率数组，第 i 项为 P(词_i | w_c)。"""
    logits = U_outside_mat @ v_c 
    return softmax(logits)

# 外层的适配器
def prob_context_given_center_adapter(w_c: str, V_center_mat, U_outside_mat, w2i):
    """给定中心词 w_c，从 V_center_mat 取 v_c 后计算 P(·|w_c)。返回形状 (|V|,) 的概率数组。"""
    v_c = V_center_mat[w2i[w_c]]
    return prob_context_given_center(v_c, U_outside_mat)

In [74]:
# 结果可视化
# 对样本对 (中心词, 上下文词) 计算 P(上下文|中心)
# 按中心词分组展示，上下文词按句子中出现顺序排列
from collections import defaultdict
grouped = defaultdict(list)
for w_c, w_o in pairs:
    grouped[w_c].append(w_o)
print("P(上下文词 | 中心词) 示例（按中心词排列，上下文按句子顺序）：\n")
for w_c in sorted(grouped.keys()):
    print(f"中心词 「{w_c}」:")
    for w_o in grouped[w_c]:
        probs = prob_context_given_center_adapter(w_c, V_center, U_outside, w2i)
        p = probs[w2i[w_o]]
        print(f"  P({w_o} | {w_c}) = {p:.6f}")
    print()

P(上下文词 | 中心词) 示例（按中心词排列，上下文按句子顺序）：

中心词 「喜欢」:
  P(我 | 喜欢) = 0.141969
  P(学习 | 喜欢) = 0.140561
  P(机器 | 喜欢) = 0.142294
  P(我 | 喜欢) = 0.141969
  P(编程 | 喜欢) = 0.144629

中心词 「学习」:
  P(我 | 学习) = 0.141422
  P(喜欢 | 学习) = 0.143283
  P(机器 | 学习) = 0.143094
  P(学习 | 学习) = 0.143186
  P(学习 | 学习) = 0.143186
  P(机器 | 学习) = 0.143094
  P(机器 | 学习) = 0.143094
  P(很 | 学习) = 0.140142
  P(有趣 | 学习) = 0.143166

中心词 「很」:
  P(机器 | 很) = 0.141889
  P(学习 | 很) = 0.142898
  P(有趣 | 很) = 0.143477

中心词 「我」:
  P(喜欢 | 我) = 0.141640
  P(学习 | 我) = 0.142891
  P(喜欢 | 我) = 0.141640
  P(编程 | 我) = 0.136025

中心词 「有趣」:
  P(学习 | 有趣) = 0.144371
  P(很 | 有趣) = 0.143567

中心词 「机器」:
  P(喜欢 | 机器) = 0.138725
  P(学习 | 机器) = 0.146580
  P(学习 | 机器) = 0.146580
  P(学习 | 机器) = 0.146580
  P(很 | 机器) = 0.144202

中心词 「编程」:
  P(我 | 编程) = 0.142039
  P(喜欢 | 编程) = 0.144975

